# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Нагай Катерина Петрівна  
**Група:** ФІТ 3-9  
**Дата:** 13.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
# Первинний аналіз набору даних
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()

print(f'Кількість оцінок: {len(ratings_df):,}')
print(f'Унікальних користувачів: {n_users:,}')
print(f'Унікальних фільмів: {n_items:,}')
print(f'Мінімальна оцінка: {ratings_df["rating"].min():.1f}')
print(f'Максимальна оцінка: {ratings_df["rating"].max():.1f}')
print(f'Середня оцінка: {ratings_df["rating"].mean():.3f}')
print('\nПропущені значення:')
display(ratings_df.isna().sum().rename('missing_count').to_frame())
print('\nРозподіл оцінок:')
display(ratings_df['rating'].value_counts().sort_index().rename_axis('rating').to_frame('count'))

Кількість оцінок: 100,000
Унікальних користувачів: 943
Унікальних фільмів: 1,682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.530

Пропущені значення:


,missing_count
user_id,0
item_id,0
rating,0
timestamp,0



Розподіл оцінок:


,count
rating,
1.0,6110
2.0,11370
3.0,27145
4.0,34174
5.0,21201


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
# Випадкове розбиття 80/20 з фіксованим random_state
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train: {train_df.shape[0]:,} оцінок ({len(train_df)/len(ratings_df):.1%})')
print(f'Test:  {test_df.shape[0]:,} оцінок ({len(test_df)/len(ratings_df):.1%})')

Train: 80,000 оцінок (80.0%)
Test:  20,000 оцінок (20.0%)


In [ ]:
# Підготовка даних у форматі Surprise
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

print('Розмір train_data:', len(train_df))
print('Розмір testset:', len(testset))
print('Діапазон оцінок:', reader.rating_scale)

Розмір train_data: 80000
Розмір testset: 20000
Діапазон оцінок: (1, 5)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=1
)
grid_search_svd.fit(train_data)
print('Пошук SVD завершено.')

Пошук SVD завершено.


In [ ]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=1
)
grid_search_knn.fit(train_data)
print('Пошук KNNBasic завершено.')

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Comput

In [ ]:
cv_summary = pd.DataFrame([
    {
        'Model': 'SVD',
        'CV RMSE': grid_search_svd.best_score['rmse'],
        'CV MAE': grid_search_svd.best_score['mae'],
        'Best params (RMSE)': str(grid_search_svd.best_params['rmse'])
    },
    {
        'Model': 'KNNBasic',
        'CV RMSE': grid_search_knn.best_score['rmse'],
        'CV MAE': grid_search_knn.best_score['mae'],
        'Best params (RMSE)': str(grid_search_knn.best_params['rmse'])
    }
]).sort_values('CV RMSE').reset_index(drop=True)

display(cv_summary)
print('SVD best RMSE params:', grid_search_svd.best_params['rmse'])
print('KNNBasic best RMSE params:', grid_search_knn.best_params['rmse'])

,Model,CV RMSE,CV MAE,Best params (RMSE)
0,SVD,0.942924,0.743406,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0..."
1,KNNBasic,1.028313,0.815422,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


SVD best RMSE params: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
KNNBasic best RMSE params: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
best_svd_params = grid_search_svd.best_params['rmse'].copy()
best_knn_params = grid_search_knn.best_params['rmse'].copy()

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params, verbose=False)

best_svd.fit(trainset)
best_knn.fit(trainset)

print('Обидві моделі навчені на повному trainset.')
print('SVD:', best_svd_params)
print('KNNBasic:', best_knn_params)

Обидві моделі навчені на повному trainset.
SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
KNNBasic: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD      — Test RMSE: {svd_rmse:.4f}; Test MAE: {svd_mae:.4f}')
print(f'KNNBasic — Test RMSE: {knn_rmse:.4f}; Test MAE: {knn_mae:.4f}')

SVD      — Test RMSE: 0.9192; Test MAE: 0.7234
KNNBasic — Test RMSE: 1.0120; Test MAE: 0.7984


In [ ]:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values('RMSE').reset_index(drop=True)

display(results)
print(f'За тестовим RMSE краща модель: {results.iloc[0]["Model"]}')

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


За тестовим RMSE краща модель: SVD


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
user_id = '196'

best_model_name = (
    'SVD' if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']
    else 'KNNBasic'
)
recommendation_model = best_svd if best_model_name == 'SVD' else best_knn
print('Модель для рекомендацій за CV RMSE:', best_model_name)

try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    inner_uid = None

if inner_uid is None:
    print(f'Користувача {user_id} немає у навчальній вибірці; персональні рекомендації неможливо сформувати.')
else:
    rated_inner_iids = {
        inner_iid
        for inner_iid, _rating in trainset.ur[inner_uid]
    }
    rated_raw_iids = {
        trainset.to_raw_iid(inner_iid)
        for inner_iid in rated_inner_iids
    }

    all_raw_iids = trainset.all_items()
    candidate_raw_iids = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in all_raw_iids
        if trainset.to_raw_iid(inner_iid) not in rated_raw_iids
    ]

    recommendations = [
        {
            'item_id': iid,
            'predicted_rating': recommendation_model.predict(user_id, iid).est
        }
        for iid in candidate_raw_iids
    ]

    top10 = (
        pd.DataFrame(recommendations)
        .sort_values('predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    display(top10)

Модель для рекомендацій за CV RMSE: SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

За результатами 3-кратної кросвалідації обрано SVD, оскільки її CV RMSE є найменшим (0.9429). Найкращі параметри SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}. Найкращі параметри KNNBasic: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}. На незалежній тестовій вибірці SVD отримала RMSE=0.9192 і MAE=0.7234, а KNNBasic — RMSE=1.0120 і MAE=0.7984. Різницю між CV і тестом слід оцінювати за таблицями cv_summary та results. Топ-10 рекомендацій сформовано для користувача 196 моделлю SVD; вже оцінені ним у навчальній вибірці фільми виключено. Обмеження підходу — холодний старт для нових користувачів і фільмів.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**